# Cross-Sell Association Rules Tuning

Bu notebook `cross_sell_association_rules` modeli için support, pair support ve scoring formüllerini dener; en iyi modeli ve tuning sonuçlarını kaydeder.

In [ ]:
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

In [ ]:
def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "requirements.txt").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Project root could not be found from current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "data" / "gold"
MODEL_DIR = PROJECT_ROOT / "artifacts" / "models"
METRICS_DIR = PROJECT_ROOT / "artifacts" / "metrics"
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "recommendation_outputs"

TRAIN_PATH = DATA_DIR / "cross_sell_train_baskets.parquet"
TEST_PATH = DATA_DIR / "cross_sell_test_baskets.parquet"
BEST_MODEL_PATH = MODEL_DIR / "cross_sell_association_rules_best.pkl"
BEST_RULES_PATH = OUTPUT_DIR / "cross_sell_association_rules_best.parquet"
TUNING_RESULTS_PATH = METRICS_DIR / "cross_sell_association_rules_tuning_results.csv"
BEST_METRICS_PATH = METRICS_DIR / "cross_sell_association_rules_best_metrics.json"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH, TEST_PATH

## Parameters

In [ ]:
MODEL_NAME = "cross_sell_association_rules"
RECOMMEND_K = 12
TOP_N_RULES_PER_ITEM = 200
MAX_BASKET_SIZE_FOR_PAIRS = 30
EVAL_SAMPLE_SIZE = 100_000
RANDOM_STATE = 42

PARAM_GRID = [
    {"min_item_support": 10, "min_pair_support": 3, "score_formula": "confidence_lift"},
    {"min_item_support": 20, "min_pair_support": 5, "score_formula": "confidence_lift"},
    {"min_item_support": 50, "min_pair_support": 10, "score_formula": "confidence_lift"},
    {"min_item_support": 20, "min_pair_support": 5, "score_formula": "confidence_support"},
    {"min_item_support": 20, "min_pair_support": 5, "score_formula": "jaccard_lift"},
]

## Load Data

In [ ]:
def normalize_articles(value):
    if value is None:
        return []
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, (list, tuple, set)):
        return [str(article) for article in value if pd.notna(article)]
    return [str(value)]


def unique_preserve_order(values):
    seen = set()
    output = []
    for value in values:
        if value not in seen:
            seen.add(value)
            output.append(value)
    return output


train_baskets = pd.read_parquet(TRAIN_PATH)
test_baskets = pd.read_parquet(TEST_PATH)

for frame in [train_baskets, test_baskets]:
    frame["t_dat"] = pd.to_datetime(frame["t_dat"])
    frame["articles"] = frame["articles"].apply(normalize_articles)
    frame["basket_size"] = frame["articles"].apply(len).astype("int32")

if EVAL_SAMPLE_SIZE is not None and len(test_baskets) > EVAL_SAMPLE_SIZE:
    eval_baskets = test_baskets.sample(EVAL_SAMPLE_SIZE, random_state=RANDOM_STATE)
else:
    eval_baskets = test_baskets

train_baskets.shape, test_baskets.shape, eval_baskets.shape

## Precompute Counts

In [ ]:
train_items = train_baskets[["articles"]].explode("articles").rename(columns={"articles": "article_id"})
item_support = train_items["article_id"].value_counts().astype("int64")
popular_items = item_support.index.tolist()

base_min_item_support = min(params["min_item_support"] for params in PARAM_GRID)
base_frequent_items = set(item_support[item_support >= base_min_item_support].index)

pair_counts = Counter()
used_basket_count = 0

for articles in train_baskets["articles"]:
    basket = sorted(set(article for article in articles if article in base_frequent_items))
    if len(basket) < 2 or len(basket) > MAX_BASKET_SIZE_FOR_PAIRS:
        continue
    used_basket_count += 1
    pair_counts.update(combinations(basket, 2))

len(item_support), len(pair_counts), used_basket_count

## Fit And Evaluate Helpers

In [ ]:
def calculate_score(score_formula, confidence, lift, jaccard, pair_count):
    if score_formula == "confidence_support":
        return confidence * np.log1p(pair_count)
    if score_formula == "jaccard_lift":
        return jaccard * np.log1p(lift) * np.log1p(pair_count)
    return confidence * np.log1p(lift) * np.log1p(pair_count)


def build_rules_dataframe(pair_counts, item_support, n_baskets, params):
    min_item_support = params["min_item_support"]
    min_pair_support = params["min_pair_support"]
    score_formula = params["score_formula"]
    rows = []

    for (left, right), pair_count in pair_counts.items():
        if pair_count < min_pair_support:
            continue

        left_count = int(item_support.get(left, 0))
        right_count = int(item_support.get(right, 0))
        if left_count < min_item_support or right_count < min_item_support:
            continue

        for antecedent, consequent, antecedent_count, consequent_count in [
            (left, right, left_count, right_count),
            (right, left, right_count, left_count),
        ]:
            support = pair_count / n_baskets
            confidence = pair_count / antecedent_count
            consequent_support = consequent_count / n_baskets
            lift = confidence / consequent_support if consequent_support > 0 else 0
            jaccard = pair_count / (antecedent_count + consequent_count - pair_count)
            score = calculate_score(score_formula, confidence, lift, jaccard, pair_count)

            rows.append(
                {
                    "antecedent": antecedent,
                    "consequent": consequent,
                    "pair_count": int(pair_count),
                    "antecedent_count": antecedent_count,
                    "consequent_count": consequent_count,
                    "support": support,
                    "confidence": confidence,
                    "lift": lift,
                    "jaccard": jaccard,
                    "score": score,
                }
            )

    rules = pd.DataFrame(rows)
    if rules.empty:
        return rules

    rules = rules.sort_values(
        ["antecedent", "score", "confidence", "lift", "pair_count"],
        ascending=[True, False, False, False, False],
    )
    rules = rules.groupby("antecedent", as_index=False).head(TOP_N_RULES_PER_ITEM)
    return rules.reset_index(drop=True)


def rules_to_lookup(rules):
    lookup = defaultdict(list)
    if rules.empty:
        return lookup

    for row in rules.itertuples(index=False):
        lookup[row.antecedent].append(
            {
                "consequent": row.consequent,
                "score": float(row.score),
                "confidence": float(row.confidence),
                "lift": float(row.lift),
                "support": float(row.support),
                "pair_count": int(row.pair_count),
            }
        )
    return lookup


def recommend_for_basket(articles, rules_by_item, popular_items, k=12):
    context = unique_preserve_order(normalize_articles(articles))
    seen = set(context)
    candidate_scores = defaultdict(float)

    for article in context:
        for rule in rules_by_item.get(article, []):
            candidate = rule["consequent"]
            if candidate not in seen:
                candidate_scores[candidate] += rule["score"]

    ranked = [
        article for article, _ in sorted(candidate_scores.items(), key=lambda item: item[1], reverse=True)
    ]

    for article in popular_items:
        if len(ranked) >= k:
            break
        if article not in seen and article not in candidate_scores:
            ranked.append(article)

    return ranked[:k]


def context_target_split(articles):
    articles = unique_preserve_order(normalize_articles(articles))
    if len(articles) < 2:
        return None, None
    split_idx = max(1, len(articles) // 2)
    return articles[:split_idx], set(articles[split_idx:])


def evaluate_baskets(test_frame, rules_by_item, popular_items, k=12):
    precisions = []
    recalls = []
    hit_rates = []

    for articles in test_frame["articles"]:
        context, target = context_target_split(articles)
        if not context or not target:
            continue

        recommendations = recommend_for_basket(context, rules_by_item, popular_items, k=k)
        hits = len(set(recommendations) & target)
        precisions.append(hits / k)
        recalls.append(hits / len(target))
        hit_rates.append(1.0 if hits > 0 else 0.0)

    return {
        "evaluated_baskets": int(len(precisions)),
        f"precision_at_{k}": float(np.mean(precisions)) if precisions else 0.0,
        f"recall_at_{k}": float(np.mean(recalls)) if recalls else 0.0,
        f"hit_rate_at_{k}": float(np.mean(hit_rates)) if hit_rates else 0.0,
    }

## Run Tuning

In [ ]:
results = []
best = None

for params in PARAM_GRID:
    rules = build_rules_dataframe(
        pair_counts=pair_counts,
        item_support=item_support,
        n_baskets=len(train_baskets),
        params=params,
    )
    rules_by_item = rules_to_lookup(rules)
    metrics = evaluate_baskets(eval_baskets, rules_by_item, popular_items, k=RECOMMEND_K)

    row = {
        **params,
        "rule_count": int(len(rules)),
        "antecedent_count": int(rules["antecedent"].nunique()) if not rules.empty else 0,
        **metrics,
    }
    results.append(row)

    selection_score = metrics[f"recall_at_{RECOMMEND_K}"]
    if best is None or selection_score > best["selection_score"]:
        best = {
            "params": params,
            "rules": rules,
            "rules_by_item": rules_by_item,
            "metrics": metrics,
            "selection_score": selection_score,
        }

tuning_results = pd.DataFrame(results).sort_values(
    [f"recall_at_{RECOMMEND_K}", f"hit_rate_at_{RECOMMEND_K}", f"precision_at_{RECOMMEND_K}"],
    ascending=False,
)

tuning_results

## Save Best Model

In [ ]:
best_model_artifact = {
    "model_name": MODEL_NAME,
    "params": {
        **best["params"],
        "recommend_k": RECOMMEND_K,
        "top_n_rules_per_item": TOP_N_RULES_PER_ITEM,
        "max_basket_size_for_pairs": MAX_BASKET_SIZE_FOR_PAIRS,
    },
    "metrics": best["metrics"],
    "rules_by_item": dict(best["rules_by_item"]),
    "popular_items": popular_items,
}

tuning_results.to_csv(TUNING_RESULTS_PATH, index=False)
best["rules"].to_parquet(BEST_RULES_PATH, index=False)

with BEST_MODEL_PATH.open("wb") as f:
    pickle.dump(best_model_artifact, f)

with BEST_METRICS_PATH.open("w", encoding="utf-8") as f:
    json.dump({"best_params": best["params"], **best["metrics"]}, f, indent=2)

print(f"Saved tuning results: {TUNING_RESULTS_PATH}")
print(f"Saved best rules: {BEST_RULES_PATH}")
print(f"Saved best model: {BEST_MODEL_PATH}")
print(f"Saved best metrics: {BEST_METRICS_PATH}")